1° Form - Colombia (English and Spanish)

In [15]:
###########################################################################
# CELL 2 - 1° FORM  - Colombia ########
###########################################################################

from __future__ import annotations
"""
pre-fill.py
-----------
Lee el tab "Resumen" de camara_extraction.xlsx y genera un Word pre-diligenciado
por empresa a partir de las plantillas en la carpeta Requirements.

Por ahora: 1° Form (English and Spanish).docx
  - Campo 1 (Razón Social / Company Name) ← razon_social  (col B del Excel)
  - Campo 2 (Nº Identificación / Tax ID)  ← nit           (col C del Excel)
"""

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   re        - Expresiones regulares (limpieza de nombres de archivo)
#   zipfile   - Lectura/escritura de archivos .docx (que son ZIPs internamente)
#   pathlib   - Manejo de rutas de archivos
#
# Terceros (instalar con pip si no están):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#   lxml      - Manipulación del XML interno del .docx  → pip install lxml
# ============================================================================

import re
import zipfile
from pathlib import Path

import pandas as pd
from lxml import etree

# ============================================================================
# RUTAS - CAMBIAR SI ES NECESARIO
# ============================================================================
EXCEL_PATH = r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output\camara_extraction.xlsx"
ANNEX_A_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\ - Colombia (English and Spanish).docx"
)
OUTPUT_DIR = r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output"
# ============================================================================

W_NS   = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
XML_NS = "http://www.w3.org/XML/1998/namespace"


def fill_placeholders(xml_bytes: bytes, replacements: dict[str, str]) -> bytes:
    """
    Reemplaza placeholders del tipo [razon_social] y [nit_cam] en el XML.

    Primera pasada: reemplazo directo sobre el texto XML.
      → Funciona cuando el placeholder está íntegro en un solo <w:t>.

    Segunda pasada: opera SOLO dentro de la región separate→end de cada campo
      FORMTEXT, sin tocar los runs del label ni la estructura del campo.
      → Cubre el caso en que Word partió el placeholder en varios runs.
    """
    # ── Pasada 1: reemplazo directo ───────────────────────────────────────────
    xml_str = xml_bytes.decode("utf-8")
    for placeholder, value in replacements.items():
        xml_str = xml_str.replace(placeholder, value)
    xml_bytes = xml_str.encode("utf-8")

    # ── Pasada 2: runs partidos dentro de campos FORMTEXT ────────────────────
    root = etree.fromstring(xml_bytes)

    for sep_run in root.xpath(
        "//w:r[w:fldChar[@w:fldCharType='separate']]",
        namespaces={"w": W_NS},
    ):
        parent = sep_run.getparent()
        siblings = list(parent)
        sep_idx = siblings.index(sep_run)

        # Recorre siblings siguientes hasta fldChar end
        value_runs: list[tuple] = []
        for elem in siblings[sep_idx + 1 :]:
            if elem.tag != f"{{{W_NS}}}r":
                continue
            if elem.xpath("w:fldChar[@w:fldCharType='end']", namespaces={"w": W_NS}):
                break
            t = elem.find(f"{{{W_NS}}}t")
            if t is not None:
                value_runs.append((elem, t))

        if not value_runs:
            continue

        full_text = "".join(t.text or "" for _, t in value_runs)
        changed = False
        for placeholder, value in replacements.items():
            if placeholder in full_text:
                full_text = full_text.replace(placeholder, value)
                changed = True

        if changed:
            _, first_t = value_runs[0]
            first_t.text = full_text
            first_t.set(f"{{{XML_NS}}}space", "preserve")
            for _, t in value_runs[1:]:
                t.text = ""

    return etree.tostring(root, xml_declaration=True, encoding="UTF-8", standalone=True)


def create_filled_docx(
    template_path: Path,
    razon_social: str,
    nit: str,
    output_path: Path,
) -> None:
    """Copia la plantilla y reemplaza [razon_social] y [nit]."""
    replacements = {
        "[razon_social]": razon_social,
        "[nit_cam]": nit,
    }
    with zipfile.ZipFile(template_path, "r") as zin:
        with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                data = zin.read(item.filename)
                if item.filename == "word/document.xml":
                    data = fill_placeholders(data, replacements)
                zout.writestr(item, data)


def safe_filename(name: str) -> str:
    """Elimina caracteres no permitidos en nombres de archivo."""
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()


def main() -> None:
    excel_path = Path(EXCEL_PATH)
    template_path = Path(ANNEX_A_TEMPLATE)
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_excel(excel_path, sheet_name="Resumen")

    processed = 0
    for _, row in df.iterrows():
        razon_social = str(row.get("razon_social", "")).strip()
        nit = str(row.get("nit", "")).strip()

        if not razon_social:
            print(f"  [SKIP] Fila sin razón social: {row.to_dict()}")
            continue

        filename = f"{safe_filename(razon_social)} - Annex A.docx"
        output_path = output_dir / filename

        create_filled_docx(template_path, razon_social, nit, output_path)
        print(f"  ✓  {filename}  |  NIT: {nit}")
        processed += 1

    print(f"\n{processed} Archivo(s) generado(s) en: {output_dir.resolve()}")


if __name__ == "__main__":
    main()


  ✓  NAME OF THE CLIENT - 1° FORM.docx  |  NIT: *** -***-594-1

1 Archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


 2° Form

In [16]:
###########################################################################
# CELL 4 - 2° Form ######
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   pathlib   - Manejo de rutas de archivos
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   fill_placeholders()   - Reemplaza [razon_social] y [nit_cam] en el XML
#   create_filled_docx()  - Genera el .docx con los valores del Excel
#   safe_filename()       - Limpia el nombre para usarlo como archivo
# ============================================================================

CONTACT_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\2° Form.docx"
)

df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print(f"  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - Contact Information Certificate.docx"
    output_path = Path(OUTPUT_DIR) / filename

    create_filled_docx(Path(CONTACT_TEMPLATE), razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO DAVIBANK S.A. - Contact Information Certificate.docx  |  NIT: 860034594-1

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


3° Form  

In [17]:
###########################################################################
# CELL 6 - 3° Form                                             #####
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   pathlib   - Manejo de rutas de archivos
#
# Terceros — NUEVAS en esta celda (instalar si no están):
#   pypdf     - Lectura y escritura de campos AcroForm en PDFs
#                                              → pip install pypdf
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   safe_filename()  - Limpia el nombre para usarlo como archivo
# ============================================================================

from pypdf import PdfReader, PdfWriter

CRS_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\3° Form.pdf"
)
BOGOTA = "Bogotá D.C."


def build_crs_fields(row: dict) -> tuple[dict, dict]:
    """Devuelve (campos_pag4, campos_pag11) a partir de una fila del Excel."""
    domicilio = str(row.get("domicilio_principal", "")).strip()
    is_bogota = domicilio == BOGOTA

    p4 = {
        "Text16": str(row.get("razon_social", "")).strip(),
        "Text17": "Colombia" if is_bogota else "",
        "Text18": str(row.get("direccion_domicilio_principal", "")).strip(),
        "Text19": "Bogota"   if is_bogota else "",
        "Text20": "Bogota"   if is_bogota else "",
        "Text21": "Colombia" if is_bogota else "",
        "Text22": "111111"   if is_bogota else "",
    }
    p11 = {
        "CJTaxResidence1": "Colombia" if is_bogota else "",
        "TIN1":            str(row.get("nit", "")).strip(),
    }
    return p4, p11


def fill_crs_pdf(template_path: Path, p4: dict, p11: dict, output_path: Path) -> None:
    """Rellena los campos del 3° Form y guarda en output_path."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    writer.update_page_form_field_values(writer.pages[3],  p4)
    writer.update_page_form_field_values(writer.pages[10], p11)
    with open(output_path, "wb") as f:
        writer.write(f)


# ── Ejecución ────────────────────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    p4, p11 = build_crs_fields(row)
    filename    = f"{safe_filename(razon_social)} - 3° Form.pdf"
    output_path = Path(OUTPUT_DIR) / filename

    fill_crs_pdf(Path(CRS_TEMPLATE), p4, p11, output_path)
    print(f"  ✓  {filename}  |  NIT: {p11['TIN1']}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO #### S.A. - 3° Form.pdf  |  NIT: 8600####-1

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


4° Form

In [18]:
###########################################################################
# CELL 8 - 4° Form                       #####
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   pathlib   - Manejo de rutas de archivos
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   fill_placeholders()   - Reemplaza [razon_social] y [nit_cam] en el XML
#   create_filled_docx()  - Genera el .docx con los valores del Excel
#   safe_filename()       - Limpia el nombre para usarlo como archivo
# ============================================================================

ESST_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\4° Form.docx"
)

df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - 4° Form.docx"
    output_path = Path(OUTPUT_DIR) / filename

    create_filled_docx(Path(ESST_TEMPLATE), razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO DAVIBANK S.A. - 4° Form.docx  |  NIT: 860034594-1

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


5° Form

In [19]:
###########################################################################
# CELL 10 - 5° Form                           #####
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   datetime  - Fecha de ejecución del script (C5)
#   shutil    - Copia del archivo template antes de modificarlo
#   pathlib   - Manejo de rutas de archivos
#
# Terceros — NUEVAS en esta celda:
#   openpyxl  - Lectura y escritura de archivos .xlsx  → pip install openpyxl
#              (normalmente ya instalada junto con pandas)
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel fuente       → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   safe_filename()  - Limpia el nombre para usarlo como archivo
# ============================================================================

import shutil
from datetime import date
from openpyxl import load_workbook

FORM_GE_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\5° Form.xlsx"
)


def parse_nit(nit_raw: str) -> tuple[str, str]:
    """
    Separa el NIT en (número, dígito_verificación).
    Ejemplo: '860034594-1' → ('860034594', '1')
    Si no hay guion devuelve (nit_raw, '').
    """
    if "-" in nit_raw:
        parts = nit_raw.split("-", 1)
        return parts[0].strip(), parts[1].strip()
    return nit_raw.strip(), ""


def fill_grandes_exposiciones(template_path: Path, row: dict, output_path: Path) -> None:
    """Copia el template y rellena las celdas C5–C11."""
    shutil.copy2(str(template_path), str(output_path))
    wb = load_workbook(str(output_path))
    ws = wb.active

    razon_social = str(row.get("razon_social", "")).strip()
    nit_raw      = str(row.get("nit", "")).strip()
    nit_num, nit_dig = parse_nit(nit_raw)

    # C5 — Fecha de diligenciamiento (DD/MM/YYYY)
    ws["C5"] = date.today()
    ws["C5"].number_format = "DD/MM/YYYY"

    # C6 — Nombre / Razón Social
    ws["C6"] = razon_social

    # C7 — Tipo ID (dropdown: NIT | TIN | OTHER)
    ws["C7"] = "NIT" if nit_raw else None

    # C8 — Dígito de verificación (dropdown: 1-9 | not applicable.)
    if nit_dig:
        try:
            ws["C8"] = int(nit_dig)
        except ValueError:
            ws["C8"] = nit_dig

    # C9 — Número de identificación (dígitos antes del guion)
    if nit_num:
        try:
            ws["C9"] = int(nit_num)
        except ValueError:
            ws["C9"] = nit_num

    # C10 — Nombre Representante Legal (placeholder)
    ws["C10"] = "[por favor diligenciar]"

    # C11 — Identificación Representante Legal (placeholder)
    ws["C11"] = "[por favor diligenciar]"

    wb.save(str(output_path))


# ── Ejecución ────────────────────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - 5° Form.xlsx"
    output_path = Path(OUTPUT_DIR) / filename

    fill_grandes_exposiciones(Path(FORM_GE_TEMPLATE), row, output_path)
    print(f"  ✓  {filename}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO DAVIBANK S.A. - 5° Form.xlsx

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


GCC - Declaracion de Veracidad

In [20]:
###########################################################################
# CELL 12 - 6° Form                             #####
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   pathlib   - Manejo de rutas de archivos
#
# Terceros — importadas directamente aquí (independiente de CELL 6):
#   pypdf     - Lectura y escritura de campos AcroForm en PDFs
#                                              → pip install pypdf
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   safe_filename()  - Limpia el nombre para usarlo como archivo
# ============================================================================

from pypdf import PdfReader, PdfWriter

GCC_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\6° Form.pdf"
)


def fill_gcc_pdf(template_path: Path, razon_social: str, nit: str, output_path: Path) -> None:
    """Rellena los campos AcroForm del 6° Form."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    writer.update_page_form_field_values(writer.pages[0], {
        "Nombre de la sociedad/empresa": razon_social,
        "(NIT)":                         nit,
    })
    with open(output_path, "wb") as f:
        writer.write(f)


# ── Ejecución ────────────────────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    nit          = str(row.get("nit", "")).strip()

    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    filename    = f"{safe_filename(razon_social)} - 6° Form.pdf"
    output_path = Path(OUTPUT_DIR) / filename

    fill_gcc_pdf(Path(GCC_TEMPLATE), razon_social, nit, output_path)
    print(f"  ✓  {filename}  |  NIT: {nit}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO DAVIBANK S.A. - 6° Form.pdf  |  NIT: 860034594-1

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


7° Form

In [21]:
###########################################################################
# CELL 14 - 7° Form            #####
###########################################################################

# ============================================================================
# LIBRERÍAS UTILIZADAS
# ============================================================================
# Estándar (incluidas con Python, no requieren instalación):
#   datetime  - Fecha de hoy en formato DD/MM/YYYY (campo "Fecha Date")
#   pathlib   - Manejo de rutas de archivos
#
# Terceros — importadas directamente aquí (independiente de CELL 6):
#   pypdf     - Lectura y escritura de campos AcroForm en PDFs
#                                              → pip install pypdf
#
# Terceros (instaladas en CELL 2, reutilizadas aquí):
#   pandas    - Lectura del archivo Excel        → pip install pandas openpyxl
#
# Funciones reutilizadas de CELL 2 (deben ejecutarse primero):
#   safe_filename()  - Limpia el nombre para usarlo como archivo
# ============================================================================

from datetime import date
from pathlib import Path

import pandas as pd
from pypdf import PdfReader, PdfWriter

KYC_TEMPLATE = (
    r"C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Requirements"
    r"\7° Form.pdf"
)
BOGOTA = "Bogotá D.C."


def build_kyc_fields(row: dict) -> dict:
    """
    Construye el diccionario de campos AcroForm a partir de una fila del Excel.

    Mapeo de campos:
      Fecha Date               ← Hoy  (DD/MM/YYYY)
      Text2  [nit_cam]         ← nit             (col C)
      Text3  [razon_social]    ← razon_social     (col B)
      Text4  [Teléfono…]       ← Teléfono comercial 1: (col F)
      Text5  [address]         ← direccion_domicilio_principal (col E)
      Text6  [pais]            ← "Colombia" si domicilio == "Bogotá D.C.", si no ""
      Código CIIU ISIC Code    ← Actividad principal Código CIIU (col G)
    """
    domicilio    = str(row.get("domicilio_principal", "")).strip()
    is_bogota    = domicilio == BOGOTA
    telefono_raw = row.get("Teléfono comercial 1:", "")
    ciiu_raw     = row.get("Actividad principal Código CIIU", "")

    return {
        "Fecha Date":            date.today().strftime("%d/%m/%Y"),
        "Text2":                 str(row.get("nit", "")).strip(),
        "Text3":                 str(row.get("razon_social", "")).strip(),
        "Text4":                 str(telefono_raw).strip() if pd.notna(telefono_raw) else "",
        "Text5":                 str(row.get("direccion_domicilio_principal", "")).strip(),
        "Text6":                 "Colombia" if is_bogota else "",
        "Código CIIU ISIC Code": str(ciiu_raw).strip() if pd.notna(ciiu_raw) else "",
    }


def fill_kyc_pdf(template_path: Path, fields: dict, output_path: Path) -> None:
    """Rellena los campos AcroForm del KYC form en todas las páginas del PDF."""
    reader = PdfReader(str(template_path))
    writer = PdfWriter()
    writer.append(reader)
    for page in writer.pages:
        writer.update_page_form_field_values(page, fields)
    with open(output_path, "wb") as f:
        writer.write(f)


# ── Ejecución ────────────────────────────────────────────────────────────────
df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")

processed = 0
for _, row in df.iterrows():
    razon_social = str(row.get("razon_social", "")).strip()
    if not razon_social:
        print("  [SKIP] Fila sin razón social")
        continue

    fields      = build_kyc_fields(row)
    filename    = f"{safe_filename(razon_social)} - KYC application form.pdf"
    output_path = Path(OUTPUT_DIR) / filename

    fill_kyc_pdf(Path(KYC_TEMPLATE), fields, output_path)
    print(f"  ✓  {filename}  |  NIT: {fields['Text2']}")
    processed += 1

print(f"\n{processed} archivo(s) generado(s) en: {Path(OUTPUT_DIR).resolve()}")

  ✓  BANCO DAVIBANK S.A. - KYC application form.pdf  |  NIT: 860034594-1

1 archivo(s) generado(s) en: C:\Users\jhern\OneDrive\Desktop\Claude Code\Cyclope_Project\Output


Folder Creation

In [ ]:
###########################################################################
# CELL 16 - Folder Creation                                           #####
###########################################################################

import re
import shutil
from pathlib import Path

import pandas as pd

# ============================================================================
# RUTAS
# ============================================================================
# EXCEL_PATH y OUTPUT_DIR ya estan definidos en CELL 2
BASE_PATH = Path(r"\\naeast.ad.jpmorganchase.com\cib2\dipls\NACIB2DIPLSSHARE00001\COB\Counterparties")
# ============================================================================


def _safe_folder_name(name: str) -> str:
    """Elimina caracteres no permitidos en nombres de carpeta en Windows."""
    return re.sub(r'[/:*?"<>|]', "_", name).strip()


def create_counterparty_folders() -> None:
    df = pd.read_excel(EXCEL_PATH, sheet_name="Resumen")
    output_dir = Path(OUTPUT_DIR)

    for _, row in df.iterrows():
        razon_social = str(row.get("razon_social", "")).strip()
        nit          = str(row.get("nit", "")).strip()

        if not razon_social:
            print(f"  [SKIP] Fila sin razon social: {row.to_dict()}")
            continue

        folder_name      = _safe_folder_name(f"{razon_social} - {nit}")
        counterparty_dir = BASE_PATH / folder_name

        if counterparty_dir.exists():
            raise FileExistsError(
                f"El folder ya existe y no se sobreescribira: {counterparty_dir}\n"
                f"Eliminalo manualmente si deseas regenerarlo."
            )

        only_mo_dir    = counterparty_dir / "Only MO"
        only_maker_dir = counterparty_dir / "Only Maker"
        only_mo_dir.mkdir(parents=True)
        only_maker_dir.mkdir(parents=True)

        files_copied = 0
        for file in output_dir.iterdir():
            if file.is_file():
                shutil.copy2(file, only_mo_dir / file.name)
                files_copied += 1

        print(f"  Folder creado  : {folder_name}")
        print(f"     Only MO    : {files_copied} archivo(s) copiado(s)")
        print(f"     Only Maker : (vacio)")


create_counterparty_folders()
